![JohnSnowLabs](https://sparknlp.org/assets/images/logo.png)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JohnSnowLabs/spark-nlp/blob/master/examples/python/transformers/onnx/HuggingFace_ONNX_in_Spark_NLP_BGE_M3.ipynb)

# Import ONNX BGE-M3 models from HuggingFace 🤗 into Spark NLP 🚀

This notebook exports [BAAI/bge-m3](https://huggingface.co/BAAI/bge-m3) for use with the `BGEM3Embeddings` annotator.

BGE-M3 is built on the `xlm-roberta-large` backbone but ships an extra `sparse_linear.pt` head that produces a per-token lexical weight. `BGEM3Embeddings` returns **both**:

- a **dense** embedding, and
- a **sparse** / lexical `{token: weight}` map (enabled with `setReturnSparseEmbeddings(True)`).

Optimum's feature-extraction export only emits `last_hidden_state`, so we export a small wrapper with `torch.onnx.export` that folds both the dense pooling and the sparse head into the graph, producing **two outputs**:

- `dense_embedding` `[batch, dim]` — CLS-pooled and L2-normalized
- `token_weights` `[batch, seq]` — the `relu`'d sparse head, one weight per token



## Export the model to ONNX (dense pooling + sparse head folded in)

In [1]:
!pip install -q  transformers==4.51.3 optimum onnx onnxruntime huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 42.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.9/160.9 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 47.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 56.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 51.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer
from huggingface_hub import hf_hub_download

MODEL_NAME = "BAAI/bge-m3"
EXPORT_PATH = f"onnx_models/{MODEL_NAME}"
# torch.onnx.export scatters raw external-tensor files next to its output. Write those into a
# throwaway RAW_EXPORT_PATH and keep EXPORT_PATH for the clean, consolidated model handed to Spark NLP.
RAW_EXPORT_PATH = f"onnx_models_raw/{MODEL_NAME}"
os.makedirs(EXPORT_PATH, exist_ok=True)
os.makedirs(RAW_EXPORT_PATH, exist_ok=True)

encoder = AutoModel.from_pretrained(MODEL_NAME).eval()
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

hidden_size = encoder.config.hidden_size  # 1024 for xlm-roberta-large
print("hidden_size:", hidden_size)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

hidden_size: 1024


In [3]:
# Load the separate sparse_linear.pt head. It is a Linear(hidden_size -> 1) whose
# state_dict holds 'weight' [1, hidden_size] and 'bias' [1].
sparse_linear_path = hf_hub_download(repo_id=MODEL_NAME, filename="sparse_linear.pt")
sparse_state = torch.load(sparse_linear_path, map_location="cpu")
print({k: tuple(v.shape) for k, v in sparse_state.items()})

sparse_linear = nn.Linear(hidden_size, 1)
sparse_linear.load_state_dict(sparse_state)
sparse_linear.eval()

sparse_linear.pt:   0%|          | 0.00/3.52k [00:00<?, ?B/s]

{'weight': (1, 1024), 'bias': (1,)}


Linear(in_features=1024, out_features=1, bias=True)

In [4]:
class BGEM3Wrapper(nn.Module):
    """Folds dense pooling and the sparse head into the graph so ONNX exposes two outputs:
    - dense_embedding: [batch, dim]  CLS token (position 0), L2-normalized
    - token_weights:   [batch, seq]  relu of the sparse_linear head, one weight per token
    """

    def __init__(self, encoder, sparse_linear):
        super().__init__()
        self.encoder = encoder
        self.sparse_linear = sparse_linear

    def forward(self, input_ids, attention_mask):
        last_hidden_state = self.encoder(
            input_ids=input_ids, attention_mask=attention_mask, return_dict=True
        ).last_hidden_state
        dense_embedding = F.normalize(last_hidden_state[:, 0], p=2, dim=-1)
        token_weights = torch.relu(self.sparse_linear(last_hidden_state)).squeeze(-1)
        return dense_embedding, token_weights


wrapper = BGEM3Wrapper(encoder, sparse_linear).eval()

In [5]:
dummy = tokenizer(["Hello world", "Bonjour le monde"], return_tensors="pt", padding=True)

raw_onnx = f"{RAW_EXPORT_PATH}/model.onnx"

torch.onnx.export(
    wrapper,
    (dummy["input_ids"], dummy["attention_mask"]),
    raw_onnx,
    input_names=["input_ids", "attention_mask"],
    output_names=["dense_embedding", "token_weights"],
    dynamic_axes={
        "input_ids": {0: "batch", 1: "seq"},
        "attention_mask": {0: "batch", 1: "seq"},
        "dense_embedding": {0: "batch"},
        "token_weights": {0: "batch", 1: "seq"},
    },
    opset_version=17,
    dynamo=False
)

/tmp/ipykernel_927/3323298366.py:5: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


xlm-roberta-large is larger than the 2GB protobuf limit in fp32, so we re-save the graph with the weights in a single external-data file.

In [6]:
import onnx
import shutil

# Read the scattered raw export back in (pulls every external tensor file into memory)...
onnx_model = onnx.load(raw_onnx, load_external_data=True)


onnx.save_model(
    onnx_model,
    f"{EXPORT_PATH}/model.onnx",
    save_as_external_data=True,
    all_tensors_to_one_file=True,
    location="_bge_m3_model.onnx_data",
    size_threshold=1024,
    convert_attribute=False,
)

shutil.rmtree(RAW_EXPORT_PATH, ignore_errors=True)

Spark NLP loads the SentencePiece tokenizer from the `assets` directory, so we move `sentencepiece.bpe.model` there.

In [7]:
tokenizer.save_pretrained(EXPORT_PATH)

!mkdir -p {EXPORT_PATH}/assets
!mv {EXPORT_PATH}/sentencepiece.bpe.model {EXPORT_PATH}/assets/
!ls -l {EXPORT_PATH}
!ls -l {EXPORT_PATH}/assets

total 2231040
drwxr-xr-x 2 root root       4096 Aug  6 23:24 assets
-rw------- 1 root root 2266824704 Aug  6 23:24 _bge_m3_model.onnx_data
-rw-r--r-- 1 root root     659337 Aug  6 23:24 model.onnx
-rw-r--r-- 1 root root        964 Aug  6 23:24 special_tokens_map.json
-rw-r--r-- 1 root root       1203 Aug  6 23:24 tokenizer_config.json
-rw-r--r-- 1 root root   17082954 Aug  6 23:24 tokenizer.json
total 4952
-rw-r--r-- 1 root root 5069051 Aug  6 23:24 sentencepiece.bpe.model


## Import and Save BGE-M3 in Spark NLP

In [9]:
!pip install -q pyspark==3.5.4 spark-nlp

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.1.0 requires pyspark[connect]~=4.0.0, but you have pyspark 3.5.4 which is incompatible.


In [ ]:
import sparknlp

spark = sparknlp.start()
print("Spark NLP version: ", sparknlp.version())
print("Apache Spark version: ", spark.version)

In [12]:
from sparknlp.annotator import BGEM3Embeddings

bge_m3 = BGEM3Embeddings.loadSavedModel(f"{EXPORT_PATH}", spark) \
    .setInputCols(["document"]) \
    .setOutputCol("bge_m3")

bge_m3.write().overwrite().save(f"bge_m3_spark_nlp")

In [14]:
from sparknlp.base import DocumentAssembler
from sparknlp.annotator import BGEM3Embeddings
from pyspark.ml import Pipeline

document_assembler = DocumentAssembler() \
    .setInputCol("text") \
    .setOutputCol("document")

bge_m3 = BGEM3Embeddings.load(f"bge_m3_spark_nlp") \
    .setInputCols(["document"]) \
    .setOutputCol("bge_m3") \
    .setReturnSparseEmbeddings(True)

pipeline = Pipeline(stages=[document_assembler, bge_m3])

data = spark.createDataFrame([
    ["BGE-M3 supports both dense and sparse retrieval."],
    ["El BGE-M3 admite recuperación densa y dispersa."],
]).toDF("text")

result = pipeline.fit(data).transform(data)

result.show(truncate=False)

+------------------------------------------------+------------------------------------------------------------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

That's it! `BGEM3Embeddings` now returns dense embeddings in `embeddings` and, when `setReturnSparseEmbeddings(True)` is set, the sparse `{token: weight}` lexical weights in `metadata`.